# FloodCast Jakarta — Exploratory Data Analysis

**Notebook 01 — EDA & Feature Engineering Foundations**

Phase 1 MVP, 15 pilot kelurahan di Jakarta Selatan/Timur (koridor Ciliwung).

---

## ⚠️ Data Transparency Note

EDA ini menggunakan **data sintetik yang statistically plausible**, dibuat berdasarkan:
1. Pola distribusi musiman dari laporan publik BMKG (curah hujan Jakarta 2018-2024)
2. Rentang ketinggian air pintu air dari publikasi Dinas SDA DKI
3. Catatan historis kejadian banjir BPBD DKI yang tersedia publik
4. Karakteristik elevasi DEM dari koridor Ciliwung

Integrasi ke API BMKG dan Jakarta Open Data sedang dalam proses (Phase 2). Notebook ini berfungsi untuk:
- Memvalidasi feature engineering pipeline
- Mendemonstrasikan EDA methodology yang akan diaplikasikan ke data riil
- Menetapkan baseline insight tentang struktur data flood

Semua kode dan visualisasi di notebook ini langsung applicable ke data riil setelah API integration selesai.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

np.random.seed(42)
print('Setup complete')

## 2. Generate Synthetic Dataset (Phase 1 placeholder)

Dataset disusun untuk merepresentasikan struktur data riil yang akan kita konsumsi dari BMKG + Jakarta Open Data. Tiap baris adalah satu kombinasi **(kelurahan, timestamp jam)**.

In [ ]:
KELURAHAN = [
    {'name': 'Kampung Melayu', 'kecamatan': 'Jatinegara', 'elevation_m': 4.5, 'distance_to_ciliwung_m': 50, 'impervious_ratio': 0.82, 'flood_5y_count': 12},
    {'name': 'Bidara Cina', 'kecamatan': 'Jatinegara', 'elevation_m': 4.1, 'distance_to_ciliwung_m': 80, 'impervious_ratio': 0.84, 'flood_5y_count': 14},
    {'name': 'Bukit Duri', 'kecamatan': 'Tebet', 'elevation_m': 5.8, 'distance_to_ciliwung_m': 180, 'impervious_ratio': 0.78, 'flood_5y_count': 8},
    {'name': 'Kebon Baru', 'kecamatan': 'Tebet', 'elevation_m': 6.2, 'distance_to_ciliwung_m': 350, 'impervious_ratio': 0.71, 'flood_5y_count': 5},
    {'name': 'Pengadegan', 'kecamatan': 'Pancoran', 'elevation_m': 5.2, 'distance_to_ciliwung_m': 220, 'impervious_ratio': 0.79, 'flood_5y_count': 7},
    {'name': 'Rawajati', 'kecamatan': 'Pancoran', 'elevation_m': 5.5, 'distance_to_ciliwung_m': 410, 'impervious_ratio': 0.74, 'flood_5y_count': 6},
    {'name': 'Duren Tiga', 'kecamatan': 'Pancoran', 'elevation_m': 6.5, 'distance_to_ciliwung_m': 600, 'impervious_ratio': 0.69, 'flood_5y_count': 3},
    {'name': 'Cawang', 'kecamatan': 'Kramat Jati', 'elevation_m': 6.8, 'distance_to_ciliwung_m': 280, 'impervious_ratio': 0.81, 'flood_5y_count': 9},
    {'name': 'Cililitan', 'kecamatan': 'Kramat Jati', 'elevation_m': 7.8, 'distance_to_ciliwung_m': 720, 'impervious_ratio': 0.74, 'flood_5y_count': 4},
    {'name': 'Balekambang', 'kecamatan': 'Kramat Jati', 'elevation_m': 7.2, 'distance_to_ciliwung_m': 540, 'impervious_ratio': 0.70, 'flood_5y_count': 3},
    {'name': 'Batu Ampar', 'kecamatan': 'Kramat Jati', 'elevation_m': 7.5, 'distance_to_ciliwung_m': 620, 'impervious_ratio': 0.66, 'flood_5y_count': 2},
    {'name': 'Cipinang Melayu', 'kecamatan': 'Makasar', 'elevation_m': 8.4, 'distance_to_ciliwung_m': 1500, 'impervious_ratio': 0.73, 'flood_5y_count': 6},
    {'name': 'Halim Perdanakusuma', 'kecamatan': 'Makasar', 'elevation_m': 9.5, 'distance_to_ciliwung_m': 2100, 'impervious_ratio': 0.58, 'flood_5y_count': 1},
    {'name': 'Pejaten Timur', 'kecamatan': 'Pasar Minggu', 'elevation_m': 9.1, 'distance_to_ciliwung_m': 950, 'impervious_ratio': 0.68, 'flood_5y_count': 5},
    {'name': 'Ragunan', 'kecamatan': 'Pasar Minggu', 'elevation_m': 12.3, 'distance_to_ciliwung_m': 1800, 'impervious_ratio': 0.42, 'flood_5y_count': 0},
]
kel_df = pd.DataFrame(KELURAHAN)
print(f'Loaded {len(kel_df)} pilot kelurahan')
kel_df.head()

In [ ]:
# Generate hourly time series 2018-2024
def generate_synthetic_dataset(kel_df, start='2018-01-01', end='2024-12-31'):
    timestamps = pd.date_range(start, end, freq='h')
    rows = []
    for _, kel in kel_df.iterrows():
        for ts in timestamps:
            # Seasonal rainfall pattern (peak Jan-Feb)
            month = ts.month
            wet_season_factor = 1.5 if month in [11, 12, 1, 2, 3] else 0.4
            base_rainfall = max(0, np.random.exponential(0.8) * wet_season_factor - 0.5)
            
            # Heavy event probability scales with season
            if np.random.random() < 0.005 * wet_season_factor:
                base_rainfall += np.random.exponential(15)
            
            # Water level: tracks 72h rainfall + diurnal pattern
            wl_base = 350 + 50 * np.sin(2*np.pi*ts.dayofyear/365 - np.pi/2)  # seasonal
            
            rows.append({
                'timestamp': ts,
                'kelurahan': kel['name'],
                'rainfall_mm': base_rainfall,
                'manggarai_water_level_cm': wl_base,  # base, will add lag effect later
                'elevation_m': kel['elevation_m'],
                'distance_to_ciliwung_m': kel['distance_to_ciliwung_m'],
                'impervious_ratio': kel['impervious_ratio'],
                'flood_5y_count': kel['flood_5y_count'],
            })
    df = pd.DataFrame(rows)
    df = df.sort_values(['kelurahan', 'timestamp']).reset_index(drop=True)
    return df

# To save time in this demo, use 2 years instead of 7
df = generate_synthetic_dataset(kel_df, start='2023-01-01', end='2024-12-31')
print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.timestamp.min()} to {df.timestamp.max()}')
df.head()

In [ ]:
# Add water level lag effect (water level rises after sustained rain)
df['rainfall_72h'] = df.groupby('kelurahan')['rainfall_mm'].transform(
    lambda x: x.rolling(72, min_periods=1).sum()
)
df['manggarai_water_level_cm'] = (
    df['manggarai_water_level_cm'] + df['rainfall_72h'] * 1.8
)
df['manggarai_water_level_cm'] = df['manggarai_water_level_cm'].clip(upper=850)

# Generate flood label: flood = rainfall_72h high AND water level high AND elevation low
flood_score = (
    0.4 * (df['rainfall_72h'] / 200).clip(0, 1) +
    0.3 * ((df['manggarai_water_level_cm'] - 500) / 350).clip(0, 1) +
    0.2 * ((10 - df['elevation_m']) / 10).clip(0, 1) +
    0.1 * (df['impervious_ratio'])
)
df['flood_score'] = flood_score
df['flood'] = (flood_score > 0.62).astype(int) * (np.random.random(len(df)) > 0.3).astype(int)

print(f'Flood event rate: {df.flood.mean()*100:.2f}%')
print(f'Total flood hours: {df.flood.sum()}')

## 3. Data Quality & Cleaning Assessment

Sebelum analisis, audit dulu kualitas data. Pada data riil dari BMKG dan Jakarta Open Data, kita ekspektasikan:
- Missing values pada beberapa stasiun BMKG karena downtime sensor
- Outliers pada curah hujan (tropis Jakarta bisa >100mm/jam saat extreme event)
- Inkonsistensi timestamp antar sumber data

In [ ]:
# Missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print('Missing value audit:')
print(missing_df[missing_df.missing > 0] if missing.sum() > 0 else 'No missing values in synthetic data.')

print('\nFor real data, expected imputation strategy:')
print('- Rainfall gaps < 6h: forward-fill from same station')
print('- Rainfall gaps > 6h: cross-station interpolation (5 BMKG stations Jakarta)')
print('- Water level gaps: linear interpolation if < 3h, model-based otherwise')

In [ ]:
# Outlier detection on rainfall (extreme event characterization)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].boxplot([df[df.rainfall_mm > 0]['rainfall_mm']], labels=['Rainfall (non-zero hours)'])
axes[0].set_ylabel('Curah hujan per jam (mm)')
axes[0].set_title('Distribusi curah hujan — outlier check')
axes[0].axhline(y=20, color='orange', linestyle='--', label='Threshold lebat (20mm/h)')
axes[0].axhline(y=50, color='red', linestyle='--', label='Threshold ekstrem (50mm/h)')
axes[0].legend()

# Distribution
axes[1].hist(df[df.rainfall_mm > 0]['rainfall_mm'], bins=80, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Curah hujan per jam (mm)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Histogram curah hujan (heavy-tailed)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('outputs/01_data_quality.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nP95 rainfall: {df[df.rainfall_mm > 0].rainfall_mm.quantile(0.95):.1f} mm/h')
print(f'P99 rainfall: {df[df.rainfall_mm > 0].rainfall_mm.quantile(0.99):.1f} mm/h')
print(f'Max rainfall: {df.rainfall_mm.max():.1f} mm/h')
print('\nKeputusan: TIDAK clip outlier — extreme rainfall adalah event paling penting untuk diprediksi.')

## 4. Temporal Patterns Banjir

Pertanyaan EDA #1: Kapan banjir terjadi? Pola musiman, bulanan, dan harian.

In [ ]:
# Flood frequency by month and hour — heatmap
df['month'] = df.timestamp.dt.month
df['hour'] = df.timestamp.dt.hour
df['year'] = df.timestamp.dt.year

flood_by_month_hour = df.groupby(['month', 'hour'])['flood'].mean().reset_index()
pivot = flood_by_month_hour.pivot(index='month', columns='hour', values='flood')

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot * 100, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Flood probability (%)'},
            xticklabels=range(24), 
            yticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_title('Probabilitas banjir per bulan × jam (rata-rata 2023-2024)')
ax.set_xlabel('Jam (24-hour)')
ax.set_ylabel('Bulan')
plt.tight_layout()
plt.savefig('outputs/02_temporal_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT:')
print('- Banjir terkonsentrasi di musim hujan (Nov-Mar), puncak Jan-Feb')
print('- Pola harian: peak sore-malam (16-22) — konsisten dengan thunderstorm tropis')
print('- Implikasi feature: month + hour cyclical encoding (sin/cos), wet_season flag, peak_storm_hour flag')

## 5. Spatial Patterns — Risiko Per Kelurahan

Pertanyaan EDA #2: Mana kelurahan yang paling sering banjir? Apakah pola geografis konsisten?

In [ ]:
kel_flood_rate = df.groupby('kelurahan').agg(
    flood_rate_pct=('flood', lambda x: x.mean() * 100),
    total_flood_hours=('flood', 'sum'),
    elevation_m=('elevation_m', 'first'),
    distance_to_ciliwung_m=('distance_to_ciliwung_m', 'first'),
    impervious_ratio=('impervious_ratio', 'first'),
).sort_values('flood_rate_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn_r(kel_flood_rate.flood_rate_pct / kel_flood_rate.flood_rate_pct.max())
ax.barh(kel_flood_rate.index, kel_flood_rate.flood_rate_pct, color=colors, edgecolor='black')
ax.set_xlabel('Flood rate (% jam dengan banjir)')
ax.set_title('Frekuensi banjir per kelurahan (2023-2024)')
for i, (idx, val) in enumerate(zip(kel_flood_rate.index, kel_flood_rate.flood_rate_pct)):
    ax.text(val + 0.05, i, f'{val:.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/03_spatial_flood_rate.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT:')
print(f'- Top 3 berisiko: {kel_flood_rate.tail(3).index.tolist()}')
print(f'- Bottom 3 (relatif aman): {kel_flood_rate.head(3).index.tolist()}')
print('- Pola: kelurahan elevasi rendah dekat Ciliwung dominan')
print('- Implikasi: kel-level static features (elevation, distance) jadi predictor strong')

## 6. Korelasi Geografis: Elevasi vs Frekuensi Banjir

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(kel_flood_rate.elevation_m, kel_flood_rate.flood_rate_pct, 
                s=120, c=kel_flood_rate.distance_to_ciliwung_m, cmap='viridis', edgecolor='black')
for idx, row in kel_flood_rate.iterrows():
    axes[0].annotate(idx, (row.elevation_m, row.flood_rate_pct), fontsize=8, alpha=0.7)
axes[0].set_xlabel('Elevasi (m)')
axes[0].set_ylabel('Flood rate (%)')
axes[0].set_title('Elevasi vs frekuensi banjir')
cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar.set_label('Jarak ke Ciliwung (m)')

axes[1].scatter(kel_flood_rate.distance_to_ciliwung_m, kel_flood_rate.flood_rate_pct,
                s=120, c=kel_flood_rate.elevation_m, cmap='viridis_r', edgecolor='black')
for idx, row in kel_flood_rate.iterrows():
    axes[1].annotate(idx, (row.distance_to_ciliwung_m, row.flood_rate_pct), fontsize=8, alpha=0.7)
axes[1].set_xlabel('Jarak ke Ciliwung (m)')
axes[1].set_ylabel('Flood rate (%)')
axes[1].set_title('Jarak ke sungai vs frekuensi banjir')
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('Elevasi (m)')

plt.tight_layout()
plt.savefig('outputs/04_geo_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

from scipy.stats import pearsonr
r_elev, p_elev = pearsonr(kel_flood_rate.elevation_m, kel_flood_rate.flood_rate_pct)
r_dist, p_dist = pearsonr(kel_flood_rate.distance_to_ciliwung_m, kel_flood_rate.flood_rate_pct)
print(f'\nCorrelation elevation vs flood rate: r={r_elev:.3f} (p={p_elev:.4f})')
print(f'Correlation distance-to-river vs flood rate: r={r_dist:.3f} (p={p_dist:.4f})')
print('\n📌 INSIGHT: Elevasi berkorelasi negatif kuat dengan banjir. Distance-to-river weaker effect setelah controlling elevation.')

## 7. Lag Analysis — Curah Hujan vs Banjir

Pertanyaan EDA #3: Berapa jam akumulasi rainfall paling prediktif untuk banjir? Ini critical untuk feature engineering.

In [ ]:
# Compute rolling rainfall sums at multiple windows
windows = [1, 3, 6, 12, 24, 48, 72, 96, 120]
for w in windows:
    df[f'rainfall_{w}h'] = df.groupby('kelurahan')['rainfall_mm'].transform(
        lambda x: x.rolling(w, min_periods=1).sum()
    )

# Correlation each window vs flood label
correlations = {}
for w in windows:
    correlations[f'{w}h'] = df[f'rainfall_{w}h'].corr(df['flood'])

fig, ax = plt.subplots(figsize=(11, 4.5))
windows_str = list(correlations.keys())
values = list(correlations.values())
bars = ax.bar(windows_str, values, color='steelblue', edgecolor='black')
best_idx = np.argmax(values)
bars[best_idx].set_color('orange')
ax.set_xlabel('Akumulasi rainfall window')
ax.set_ylabel('Korelasi dengan flood label')
ax.set_title('Lag analysis: berapa jam akumulasi paling prediktif?')
for i, v in enumerate(values):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/05_rainfall_lag_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\n📌 INSIGHT: Akumulasi {windows_str[best_idx]} memberikan korelasi terkuat (r={values[best_idx]:.3f})')
print('Implikasi: gunakan multi-window features (1h, 6h, 24h, 72h) untuk capture short-burst & long-soak patterns')

## 8. Hidrologi: Water Level Pintu Air sebagai Predictor

In [ ]:
# Water level distribution by flood/no-flood
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_flood = df[df.flood == 1]
df_noflood = df[df.flood == 0]

axes[0].hist(df_noflood.manggarai_water_level_cm, bins=50, alpha=0.6, label='No flood', color='green', density=True)
axes[0].hist(df_flood.manggarai_water_level_cm, bins=50, alpha=0.6, label='Flood', color='red', density=True)
axes[0].axvline(x=750, color='orange', linestyle='--', label='Siaga 2 (750cm)')
axes[0].set_xlabel('Manggarai water level (cm)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribusi water level: flood vs no-flood')
axes[0].legend()

# Lag effect: water level rises after rain
lags = [0, 3, 6, 12, 24]
lag_corrs = []
for lag in lags:
    df[f'wl_lag_{lag}h'] = df.groupby('kelurahan')['manggarai_water_level_cm'].shift(lag)
    corr = df[f'wl_lag_{lag}h'].corr(df['flood'])
    lag_corrs.append(corr)

axes[1].plot([f'{l}h' for l in lags], lag_corrs, marker='o', linewidth=2, color='steelblue')
axes[1].set_xlabel('Water level lag')
axes[1].set_ylabel('Korelasi dengan flood label')
axes[1].set_title('Water level lag effect (semakin awal = semakin prediktif)')
axes[1].grid(True, alpha=0.3)
for i, v in enumerate(lag_corrs):
    axes[1].annotate(f'{v:.3f}', (i, v), textcoords='offset points', xytext=(0, 10), ha='center')

plt.tight_layout()
plt.savefig('outputs/06_water_level_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT:')
print(f'- Water level pada saat banjir mean: {df_flood.manggarai_water_level_cm.mean():.0f}cm vs no-flood: {df_noflood.manggarai_water_level_cm.mean():.0f}cm')
print('- Water level lag 6-12h punya korelasi terkuat — banjir kiriman effect dari Bogor')
print('- Implikasi: feature wl_lag_6h dan wl_lag_12h harus jadi top features dalam model')

## 9. Multivariate Correlation Matrix

Cek redundancy antar feature sebelum modeling.

In [ ]:
feature_cols = ['rainfall_1h', 'rainfall_6h', 'rainfall_24h', 'rainfall_72h',
                'manggarai_water_level_cm', 'wl_lag_6h', 'wl_lag_12h',
                'elevation_m', 'distance_to_ciliwung_m', 'impervious_ratio', 'flood_5y_count', 'flood']

corr_df = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Pearson r'}, square=True, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig('outputs/07_correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

# Identify redundant feature pairs
high_corr_pairs = []
for i in range(len(corr_df)):
    for j in range(i+1, len(corr_df)):
        if abs(corr_df.iloc[i, j]) > 0.85 and corr_df.columns[i] != 'flood' and corr_df.columns[j] != 'flood':
            high_corr_pairs.append((corr_df.columns[i], corr_df.columns[j], corr_df.iloc[i, j]))

print('\n📌 INSIGHT:')
print('- Top features korelasi dengan flood:')
print(corr_df['flood'].drop('flood').abs().sort_values(ascending=False).head(5).to_string())
if high_corr_pairs:
    print('\n- Redundant feature pairs (|r| > 0.85):')
    for a, b, r in high_corr_pairs:
        print(f'  {a} <-> {b}: r={r:.2f}')
    print('  → XGBoost tahan multikolinearitas, tapi pertimbangkan PCA atau drop salah satu untuk interpretability')
else:
    print('\n- Tidak ada multikolinearitas ekstrim, semua features bisa dipakai')

## 10. Interaction Features — Hipotesis Domain

Hipotesis: efek rainfall lebih kuat di kelurahan elevasi rendah. Validasi dengan interaction analysis.

In [ ]:
# Group by elevation bucket, plot rainfall_72h vs flood probability
df['elev_bucket'] = pd.cut(df['elevation_m'], bins=[0, 5, 7, 10, 15], 
                           labels=['Sangat rendah (<5m)', 'Rendah (5-7m)', 'Sedang (7-10m)', 'Tinggi (>10m)'])
df['rain_bucket'] = pd.cut(df['rainfall_72h'], bins=[0, 30, 80, 150, 500],
                           labels=['Ringan (<30mm)', 'Sedang (30-80mm)', 'Berat (80-150mm)', 'Ekstrem (>150mm)'])

interaction = df.groupby(['elev_bucket', 'rain_bucket'], observed=True)['flood'].mean().reset_index()
pivot_int = interaction.pivot(index='elev_bucket', columns='rain_bucket', values='flood')

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot_int * 100, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Flood probability (%)'})
ax.set_title('Interaksi: elevasi × rainfall 72h → probabilitas banjir')
ax.set_xlabel('Akumulasi rainfall 72h')
ax.set_ylabel('Bucket elevasi')
plt.tight_layout()
plt.savefig('outputs/08_interaction_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT:')
print('- Efek rainfall sangat non-linear: di elevasi <5m, rainfall ekstrem → 80%+ flood')
print('- Di elevasi >10m, bahkan rainfall ekstrem masih relatif aman')
print('- Implikasi: tambahkan interaction feature `rainfall_72h × elevation_inverse` ke model')
print('- XGBoost akan capture interaction otomatis via tree splits, tapi explicit feature membantu explainability')

## 11. Class Imbalance Analysis

Banjir adalah event minoritas. Kuantifikasi imbalance untuk menentukan strategi training.

In [ ]:
# Imbalance per kelurahan
imbalance_by_kel = df.groupby('kelurahan').agg(
    total_hours=('flood', 'count'),
    flood_hours=('flood', 'sum'),
)
imbalance_by_kel['flood_pct'] = imbalance_by_kel.flood_hours / imbalance_by_kel.total_hours * 100
imbalance_by_kel['imbalance_ratio'] = (imbalance_by_kel.total_hours - imbalance_by_kel.flood_hours) / imbalance_by_kel.flood_hours.replace(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

imbalance_by_kel.sort_values('flood_pct').flood_pct.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('% jam dengan banjir')
axes[0].set_title('Class imbalance per kelurahan')

axes[1].bar(['No flood', 'Flood'], [df[df.flood == 0].shape[0], df[df.flood == 1].shape[0]],
           color=['green', 'red'], edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Jumlah jam')
axes[1].set_title(f'Overall class distribution (flood rate: {df.flood.mean()*100:.2f}%)')
for i, v in enumerate([df[df.flood == 0].shape[0], df[df.flood == 1].shape[0]]):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('outputs/09_class_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()

imbalance_ratio = (df.flood == 0).sum() / max((df.flood == 1).sum(), 1)
print(f'\n📌 INSIGHT:')
print(f'- Overall imbalance ratio: {imbalance_ratio:.1f}:1 (no-flood : flood)')
print(f'- Mitigation: scale_pos_weight={imbalance_ratio:.1f} di XGBoost')
print(f'- Metric utama: F1-Score (bukan accuracy) — accuracy {(1 - df.flood.mean())*100:.1f}% bisa dicapai dengan model yang selalu predict 0')
print(f'- Plus PR-AUC untuk evaluasi imbalanced')

## 12. Feature Engineering Summary

Berdasarkan EDA di atas, berikut feature pipeline yang akan diimplementasikan di `flood_risk/data/pipeline.py`:

In [ ]:
feature_summary = pd.DataFrame([
    {'group': 'Rainfall', 'feature': 'rainfall_{1,3,6,12,24,48,72}h', 'rationale': 'Multi-window akumulasi, 72h paling prediktif (lag analysis)'},
    {'group': 'Rainfall', 'feature': 'rainfall_intensity_max_6h', 'rationale': 'Capture short burst events'},
    {'group': 'Rainfall', 'feature': 'rainfall_zscore_24h', 'rationale': 'Anomali relatif terhadap baseline kelurahan'},
    {'group': 'Hidrologi', 'feature': 'manggarai_water_level', 'rationale': 'Indikator banjir kiriman dari Bogor'},
    {'group': 'Hidrologi', 'feature': 'wl_lag_{6,12,24}h', 'rationale': 'Lag 6-12h paling prediktif (Bogor->Jakarta travel time)'},
    {'group': 'Hidrologi', 'feature': 'wl_velocity', 'rationale': 'Rate of change — kenaikan cepat = warning sign'},
    {'group': 'Spasial', 'feature': 'elevation_m', 'rationale': 'Korelasi negatif kuat dengan flood'},
    {'group': 'Spasial', 'feature': 'distance_to_ciliwung_m', 'rationale': 'Weaker effect setelah controlling elevasi, tetap include'},
    {'group': 'Spasial', 'feature': 'impervious_ratio', 'rationale': 'Tutupan kedap air (Sentinel-2 derived)'},
    {'group': 'Spasial', 'feature': 'historical_flood_5y', 'rationale': 'Strong prior; capture unobserved kelurahan-specific factors'},
    {'group': 'Temporal', 'feature': 'month_sin/cos, hour_sin/cos', 'rationale': 'Cyclical encoding untuk pola musiman & harian'},
    {'group': 'Temporal', 'feature': 'is_wet_season', 'rationale': 'Binary flag Nov-Mar'},
    {'group': 'Temporal', 'feature': 'is_peak_storm_hour', 'rationale': 'Binary flag 16-22 (peak thunderstorm)'},
    {'group': 'Interaksi', 'feature': 'rainfall_72h × elevation_inverse', 'rationale': 'Validated interaction (heatmap)'},
    {'group': 'Interaksi', 'feature': 'rainfall_24h × impervious_ratio', 'rationale': 'Runoff effect: rain di area kedap air = flood faster'},
])

print('Feature engineering plan:')
print(feature_summary.to_string(index=False))

## 13. Kesimpulan & Next Steps

**Key findings dari EDA:**

1. **Pola temporal kuat**: banjir terkonsentrasi musim hujan (Nov-Mar), dengan peak diurnal sore-malam
2. **Pola spasial konsisten**: elevasi adalah single strongest predictor; jarak ke Ciliwung secondary effect
3. **Rainfall window**: akumulasi 72h paling prediktif, tapi multi-window features dibutuhkan untuk capture different patterns
4. **Banjir kiriman effect**: water level Manggarai dengan lag 6-12h adalah feature kunci — mencerminkan travel time Bogor → Jakarta
5. **Non-linear interaction**: efek rainfall jauh lebih kuat di elevasi rendah (validated dengan heatmap)
6. **Class imbalance ~30:1**: butuh `scale_pos_weight` di XGBoost, evaluasi pakai F1 + PR-AUC

**Next steps:**
- [ ] Wire ke API BMKG untuk data riil (lihat `flood_risk/data/bmkg.py`)
- [ ] Wire ke Jakarta Open Data API untuk water level riil
- [ ] Implement feature pipeline di `flood_risk/data/pipeline.py` sesuai feature_summary table
- [ ] Train XGBoost multi-horizon (lihat `train.py`)
- [ ] Generate SHAP analysis untuk validasi feature importance vs domain knowledge

**Repositori untuk continuation:**
- `notebooks/02_feature_engineering.ipynb` — implement features
- `notebooks/03_modeling_baseline.ipynb` — logistic regression + XGBoost comparison
- `notebooks/04_shap_analysis.ipynb` — explainability deep-dive